<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [50]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    mnist_fashion = fetch_openml('Fashion-MNIST', version=1, return_X_y=False, as_frame=False)
    X = mnist_fashion.data
    y = mnist_fashion.target.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = load_data(seed=42)

Não é necessário normalizar, pois as decisões em modelos baseados em árvores de decisões são feitas baseadas em limiares, não em magnitude.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [52]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)
    return model

random_forest = train_random_forest(X_train, y_train)
adaboost = train_adaboost(X_train, y_train)

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [53]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

rf_train_pred = random_forest.predict(X_train)
rf_test_pred = random_forest.predict(X_test)
ab_train_pred = adaboost.predict(X_train)
ab_test_pred = adaboost.predict(X_test)

print("Random Forest:")
print(f"  Acuracia treino: {accuracy_score(y_train, rf_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, rf_test_pred)}")

print("\nAdaBoost:")
print(f"  Acuracia treino: {accuracy_score(y_train, ab_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, ab_test_pred)}")

Random Forest:
  Acuracia treino: 1.0
  Acuracia teste:  0.8818571428571429

AdaBoost:
  Acuracia treino: 0.5756607142857143
  Acuracia teste:  0.5728571428571428


**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [54]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed=seed)
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed=seed)
    return evaluate(model, X_test, y_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = list(range(5, 51, 5))

print("=== Random Forest ===")
print(f"{'Profundidade':<15} {'Acc Treino':<15} {'Acc Teste':<15}")
prev_acc_test = 0
for d in depths:
    rf = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42)
    rf.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, rf.predict(X_train))
    acc_test = accuracy_score(y_test, rf.predict(X_test))
    print(f"{d:<15} {acc_train:<15.4f} {acc_test:<15.4f}")
    if acc_test < prev_acc_test:
        print(f"Overfitting detectado na profundidade {d} (acc teste caiu de {prev_acc_test:.4f} para {acc_test:.4f})")
        break
    if acc_train >= 1.0:
        print(f"Acuracia de treino maxima atingida na profundidade {d}, parando busca.")
        break
    prev_acc_test = acc_test

print("\n=== AdaBoost ===")
print(f"{'Profundidade':<15} {'Acc Treino':<15} {'Acc Teste':<15}")
prev_acc_test = 0
for d in depths:
    ab = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=d),
        n_estimators=100, random_state=42
    )
    ab.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, ab.predict(X_train))
    acc_test = accuracy_score(y_test, ab.predict(X_test))
    print(f"{d:<15} {acc_train:<15.4f} {acc_test:<15.4f}")
    if acc_test < prev_acc_test:
        print(f"Overfitting detectado na profundidade {d} (acc teste caiu de {prev_acc_test:.4f} para {acc_test:.4f})")
        break
    if acc_train >= 1.0:
        print(f"Acuracia de treino maxima atingida na profundidade {d}, parando busca.")
        break
    prev_acc_test = acc_test

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

A árvore alcança 100% no treino quando não tem profundidade máxima, pois o modelo se adapta aos dados de treino, causando overfitting.

Alcançam overfitting nas profundidades 35 para o random forest e 20 para o adaboost

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [56]:
from sklearn.metrics import classification_report

X_tr, X_te, y_tr, y_te = load_data(seed=42)

rf = train_random_forest(X_tr, y_tr)
print("Random Forest:")
print(classification_report(y_te, rf.predict(X_te)))

print("\n=========================================================\n")

ab = train_adaboost(X_tr, y_tr)
print("AdaBoost:")
print(classification_report(y_te, ab.predict(X_te)))

Random Forest:
              precision    recall  f1-score   support

           0       0.82      0.87      0.84      1400
           1       0.99      0.96      0.98      1400
           2       0.78      0.82      0.80      1400
           3       0.87      0.92      0.90      1400
           4       0.78      0.84      0.81      1400
           5       0.97      0.97      0.97      1400
           6       0.72      0.57      0.64      1400
           7       0.94      0.94      0.94      1400
           8       0.97      0.96      0.97      1400
           9       0.95      0.96      0.95      1400

    accuracy                           0.88     14000
   macro avg       0.88      0.88      0.88     14000
weighted avg       0.88      0.88      0.88     14000



AdaBoost:
              precision    recall  f1-score   support

           0       0.77      0.58      0.66      1400
           1       0.99      0.76      0.86      1400
           2       0.41      0.79      0.54      14

O modelo que apresentou melhor desempenho inicial foi o random forest, pois apresentou melhores métricas(accuracy, precision, recall e f1-score) que o adaboost.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [57]:
print("Random Forest (seed=42):")
print(f"  Acuracia: {run_pipeline('rf', 42):.4f}")
print("\nRandom Forest (seed=7):")
print(f"  Acuracia: {run_pipeline('rf', 7):.4f}")

print("\n=========================================================\n")

print("AdaBoost (seed=42):")
print(f"  Acuracia: {run_pipeline('ab', 42):.4f}")
print("\nAdaBoost (seed=7):")
print(f"  Acuracia: {run_pipeline('ab', 7):.4f}")

Random Forest (seed=42):
  Acuracia: 0.8819

Random Forest (seed=7):
  Acuracia: 0.8801


AdaBoost (seed=42):
  Acuracia: 0.5729

AdaBoost (seed=7):
  Acuracia: 0.5009


Os resultados variaram para ambos modelos.

Sim, a seed garante a reprodutibilidade dos modelos, pois gera sempre a mesma sequência de números pseudoaleatórios que os modelos utilizam para setar parâmetros e realizar funções em seu treinamento

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [58]:
print("Random Forest:")
print(f"  Acuracia treino: {accuracy_score(y_train, rf_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, rf_test_pred)}")

print("\nAdaBoost:")
print(f"  Acuracia treino: {accuracy_score(y_train, ab_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, ab_test_pred)}")

Random Forest:
  Acuracia treino: 1.0
  Acuracia teste:  0.8818571428571429

AdaBoost:
  Acuracia treino: 0.5756607142857143
  Acuracia teste:  0.5728571428571428


Sim. No caso estudado, o random forest apresenta overfitting e o adaboost não apresenta.

No caso estudado, o random forest sofre mais com overfitting por utilizar árvores sem limites de profundidade.

OBS: Na questão 4, conseguimos ver que ambos os modelos podem apresentar overfitting dependendo da limitação da profundidade das árvores utilizadas.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [59]:
print("Random Forest sem variação de hiperparâmetro:")
print(classification_report(y_test, rf_test_pred))

print(f"  Acuracia treino: {accuracy_score(y_train, rf_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, rf_test_pred)}")

print("\n=========================================================\n")

#Por padrão utiliza max_depth sem limites, passando a base pode-se limitar a profundidade.
random_forest_v2 = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15)
random_forest_v2.fit(X_train, y_train)
rf_v2_pred = random_forest_v2.predict(X_test)

rf_v2_train_pred = random_forest_v2.predict(X_train)

print("Random Forest com variação de hiperparâmetro:")
print(classification_report(y_test, rf_v2_pred))

print(f"  Acuracia treino: {accuracy_score(y_train, rf_v2_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, rf_v2_pred)}")

print("\n=========================================================\n")

print("Adaboost sem variação de hiperparâmetro:")
print(classification_report(y_test, ab_test_pred))
print(f"  Acuracia treino: {accuracy_score(y_train, ab_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, ab_test_pred)}")

print("\n=========================================================\n")

#Aqui o problema é contrário, por padrão o max_depth é 1.
adaboost_v2 = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3), n_estimators=100, random_state=42)
adaboost_v2.fit(X_train, y_train)
ab_v2_pred = adaboost_v2.predict(X_test)

ab_v2_train_pred = adaboost_v2.predict(X_train)

print("Adaboost com variação de hiperparâmetro:")
print(classification_report(y_test, ab_v2_pred))

print(f"  Acuracia treino: {accuracy_score(y_train, ab_v2_train_pred)}")
print(f"  Acuracia teste:  {accuracy_score(y_test, ab_v2_pred)}")


Random Forest sem variação de hiperparâmetro:
              precision    recall  f1-score   support

           0       0.82      0.87      0.84      1400
           1       0.99      0.96      0.98      1400
           2       0.78      0.82      0.80      1400
           3       0.87      0.92      0.90      1400
           4       0.78      0.84      0.81      1400
           5       0.97      0.97      0.97      1400
           6       0.72      0.57      0.64      1400
           7       0.94      0.94      0.94      1400
           8       0.97      0.96      0.97      1400
           9       0.95      0.96      0.95      1400

    accuracy                           0.88     14000
   macro avg       0.88      0.88      0.88     14000
weighted avg       0.88      0.88      0.88     14000

  Acuracia treino: 1.0
  Acuracia teste:  0.8818571428571429


Random Forest com variação de hiperparâmetro:
              precision    recall  f1-score   support

           0       0.82      0.

O adaboost é mais sensível, pois sua acuracia saltou de 57% para 70% ao aumentar a profundidade da árvore utilizada, enquanto o random forest variou apenas de 88.2% para 87.5%.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. Não, pois a acurácia é uma média geral da taxa de acerto, não mostrando o desempenho por classe individual, dado este que o recall, F1 e precision revelam.

2. Utilizando a seed (random_state) podemos garantir que os resultados são reproduzíveis, porém para garantir sua validade deveríamos realizar validação cruzada ou testar um número maior de seeds.

3. Fora um pequeno ajuste na questão 8, não foi feito ajuste de hiperparêmetros, então os modelos não estão conseguindo mostrar seu melhor desempenho possível com este dataset. O experimento também não considera tempo de treinamento nem custo computacional, parâmetros extremamente importantes no uso destes modelos.

4. O pipeline permite garantir a reprodutibilidade por meio da seed e segue uma estrutura bem definida, porêm para ser mais confiável, deveria permitir o ajuste de hiperparâmetros individuais de cada modelo e realizar análise de métricas para garantir a qualidade do modelo gerado.